# Lip-sync on a free GPU (LatentSync 1.5)

Makes the person in a video speak any audio. Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

LatentSync pins torch 2.5.1, which has no wheels for Colab's Python 3.13, so it runs in its own Python 3.10 environment.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
pip install -q uv
cd /content
[ -d LatentSync ] || git clone -q https://github.com/bytedance/LatentSync.git
cd LatentSync
uv venv -q -p 3.10 /content/ls310
uv pip install -q --python /content/ls310/bin/python -r requirements.txt
uv pip install -q --python /content/ls310/bin/python "huggingface_hub[hf_xet]<1"
/content/ls310/bin/hf download ByteDance/LatentSync-1.5 latentsync_unet.pt whisper/tiny.pt --local-dir checkpoints
ls -la checkpoints checkpoints/whisper
/content/ls310/bin/python -c "import torch;print(torch.__version__, torch.cuda.is_available())"

## Upload the video (face) and the audio (voice)

In [ ]:
from google.colab import files
import os
os.makedirs('/content/in', exist_ok=True)
up = files.upload()
for name, data in up.items():
    open('/content/in/' + name, 'wb').write(data)
print(os.listdir('/content/in'))

## Lip-sync (about 1 min per 10 s of video on a T4)

In [ ]:
import glob
VIDEO = sorted(glob.glob('/content/in/*.mp4'))[0]
AUDIO = sorted(glob.glob('/content/in/*.wav') + glob.glob('/content/in/*.mp3'))[0]
print(VIDEO, AUDIO)
!cd /content/LatentSync && /content/ls310/bin/python -m scripts.inference --unet_config_path configs/unet/stage2.yaml --inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps 25 --guidance_scale 1.5 --enable_deepcache --video_path "$VIDEO" --audio_path "$AUDIO" --video_out_path /content/lipsync_out.mp4
!ls -la /content/lipsync_out.mp4

In [ ]:
from google.colab import files
files.download('/content/lipsync_out.mp4')